In [ ]:
import pandas as pd
import numpy as np
import math
from typing import List, Dict, Tuple

# Global constants
EDU_RANK = {'Bac': 0, 'Bachelor': 1, 'License': 2, 'Master': 3, 'PhD': 4}

def load_data() -> Tuple[pd.DataFrame, pd.DataFrame]:
    try:
        seekers_df = pd.read_csv('emplo.csv')[[
            'gender', 'birth_date', 'age', 'city', 'department',
            'sector', 'years_experience', 'highest_education',
            'contract_type', 'technical_skills', 'language_proficiency',
            'education_history', 'edu_value', 'salary'
        ]]
    except FileNotFoundError:
        seekers_df = pd.DataFrame([{
            'gender': 'Male', 'birth_date': '1990-05-15', 'age': 33,
            'city': 'Algiers', 'department': 'IT', 'sector': 'Technology',
            'years_experience': 5, 'highest_education': 'Master',
            'contract_type': 'Full-time', 'technical_skills': 'Python, SQL, Spark',
            'language_proficiency': 'English, French', 'education_history': 'University of Algiers',
            'edu_value': 3, 'salary': 70000
        }])

    jobs_df = pd.DataFrame([
        {'job_id': 'DEV-001', 'required_skills': ['HYSYS', 'Communication', 'Problem Solving'],
         'min_education': 'License', 'min_experience': 3, 'salary_offer': 75000,
         'location': 'Algiers', 'sector': 'Energy & Petroleum', 'contract_type': 'CDD'},
        {'job_id': 'DATA-002', 'required_skills': ['python', 'Problem Solving', 'Teamwork'],
         'min_education': 'Bachelor', 'min_experience': 2, 'salary_offer': 65000,
         'location': 'Oran', 'sector': 'Data Science', 'contract_type': 'CDD'}
    ])
    
    return seekers_df, jobs_df

# ---------- 2. Core Matching Functions ----------
def preprocess_data(seekers_df: pd.DataFrame, jobs_df: pd.DataFrame) -> Tuple:
    """Preprocess and normalize all data for matching"""
    # Education ranking
    seekers_df['edu_rank'] = seekers_df['highest_education'].map(EDU_RANK).fillna(-1).astype(int)
    jobs_df['edu_rank'] = jobs_df['min_education'].map(EDU_RANK).fillna(-1).astype(int)

    # Skills processing
    seekers_df['technical_skills'] = seekers_df['technical_skills'].fillna('').str.lower()
    seeker_skills = [set(s.split(', ')) for s in seekers_df['technical_skills']]
    job_skills = [set(map(str.lower, req)) for req in jobs_df['required_skills']]

    # Sector and contract type normalization
    all_sectors = pd.Categorical(seekers_df['sector'].tolist() + jobs_df['sector'].tolist())
    seek_sector_codes = pd.Categorical(seekers_df['sector'], categories=all_sectors.categories).codes
    job_sector_codes = pd.Categorical(jobs_df['sector'], categories=all_sectors.categories).codes

    contract_types = pd.Categorical(seekers_df['contract_type'].tolist() + jobs_df['contract_type'].tolist())
    seek_cont_codes = pd.Categorical(seekers_df['contract_type'], categories=contract_types.categories).codes
    job_cont_codes = pd.Categorical(jobs_df['contract_type'], categories=contract_types.categories).codes

    return seekers_df, jobs_df, seeker_skills, job_skills, seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes

def calculate_features(seekers_df, jobs_df, seeker_skills, job_skills, seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes) -> np.ndarray:
    """Calculate all feature scores between seekers and jobs"""
    N_seek = len(seekers_df)
    N_job = len(jobs_df)
    F_raw = np.zeros((N_seek, N_job, 7), dtype=np.float32)

    # Skill TF-IDF calculation
    all_skills = [skill for skills in seekers_df['technical_skills'] for skill in skills.split(', ') if skill]
    skill_counts = pd.Series(all_skills).value_counts()
    total_docs = N_seek + N_job
    skill_idf = {s: np.log((total_docs + 1)/(cnt + 1)) + 1 for s, cnt in skill_counts.items()}

    # Salary normalization
    all_salaries = np.concatenate([seekers_df['salary'].values, jobs_df['salary_offer'].values])
    max_sal = np.percentile(all_salaries, 95)
    min_sal = np.percentile(all_salaries, 5)

    for i in range(N_seek):
        for j in range(N_job):
            # Skill match (40%)
            common_skills = seeker_skills[i] & job_skills[j]
            missing_skills = job_skills[j] - seeker_skills[i]
            penalty = 1 - (len(missing_skills) / len(job_skills[j]))
            match_score = sum(skill_idf.get(s, 0) for s in common_skills)
            total_score = sum(skill_idf.get(s, 0) for s in job_skills[j])
            F_raw[i,j,0] = penalty * (match_score / total_score if total_score > 0 else 0)

            # Experience (15%)
            F_raw[i,j,1] = min(seekers_df.at[i,'years_experience'] / max(jobs_df.at[j,'min_experience'], 1), 1.5)

            # Salary (15%)
            salary_diff = abs(jobs_df.at[j,'salary_offer'] - seekers_df.at[i,'salary'])
            F_raw[i,j,2] = 1 - np.log1p(salary_diff) / np.log1p(max_sal - min_sal)

            # Education (10%)
            F_raw[i,j,3] = seekers_df.at[i,'edu_rank'] / max(EDU_RANK.values())

            # Sector (10%)
            F_raw[i,j,4] = (seek_sector_codes[i] == job_sector_codes[j])

            # Contract (5%)
            F_raw[i,j,5] = (seek_cont_codes[i] == job_cont_codes[j])

            # Education value (5%)
            F_raw[i,j,6] = seekers_df.at[i,'edu_value'] / 20

    # Min-Max Normalization
    feature_mins = F_raw.min(axis=(0,1))
    feature_maxs = F_raw.max(axis=(0,1))
    F = (F_raw - feature_mins) / (feature_maxs - feature_mins + 1e-8)
    
    return F

# ---------- 3. Genetic Algorithm Implementation ----------
def initialize_population(pop_size: int, N_seek: int, valid_jobs: np.ndarray) -> np.ndarray:
    """Initialize population with valid job assignments"""
    pop = np.empty((pop_size, N_seek), dtype=int)
    for i in range(N_seek):
        choices = np.append(np.where(valid_jobs[i])[0], -1)  # -1 means no job
        pop[:,i] = np.random.choice(choices, pop_size)
    return pop

def calculate_fitness(pop: np.ndarray, F: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Calculate fitness scores for population"""
    scores = np.zeros(pop.shape[0])
    for k in range(pop.shape[0]):
        for i in range(pop.shape[1]):
            j = pop[k,i]
            if j >= 0:  # Only count if assigned to a valid job
                scores[k] += F[i,j].dot(weights)
        # Penalize unmatched candidates
        scores[k] -= 0.1 * np.sum(pop[k] == -1)
    return scores

def tournament_selection(pop: np.ndarray, fitness: np.ndarray, tournament_size: int = 3) -> np.ndarray:
    """Select parents using tournament selection"""
    selected = np.empty_like(pop)
    for i in range(pop.shape[0]):
        contenders = np.random.choice(len(fitness), tournament_size, replace=False)
        winner = pop[contenders[np.argmax(fitness[contenders])]]
        selected[i] = winner
    return selected

def crossover(parent1: np.ndarray, parent2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Perform crossover between two parents"""
    if len(parent1) <= 1:
        return parent1.copy(), parent2.copy()
    cut = np.random.randint(1, len(parent1))
    child1 = np.concatenate([parent1[:cut], parent2[cut:]])
    child2 = np.concatenate([parent2[:cut], parent1[cut:]])
    return child1, child2

def mutate(chromosome: np.ndarray, mutation_rate: float, valid_jobs: np.ndarray) -> np.ndarray:
    """Mutate chromosome with given rate"""
    for i in range(len(chromosome)):
        if np.random.rand() < mutation_rate:
            choices = np.append(np.where(valid_jobs[i])[0], -1)
            chromosome[i] = np.random.choice(choices)
    return chromosome

def run_genetic_algorithm(F: np.ndarray, weights: np.ndarray, valid_jobs: np.ndarray,
                         pop_size: int = 50, generations: int = 100,
                         mutation_rate: float = 0.1, elite_size: int = 2) -> Tuple[np.ndarray, float]:
    """Run the complete genetic algorithm"""
    N_seek = F.shape[0]
    pop = initialize_population(pop_size, N_seek, valid_jobs)
    best_score = -np.inf
    best_solution = None
    
    for gen in range(generations):
        # Evaluate fitness
        fitness = calculate_fitness(pop, F, weights)
        
        # Track best solution
        current_best = np.argmax(fitness)
        if fitness[current_best] > best_score:
            best_score = fitness[current_best]
            best_solution = pop[current_best].copy()
        
        # Selection
        selected = tournament_selection(pop, fitness)
        
        # Crossover
        new_pop = []
        for i in range(0, pop_size - elite_size, 2):
            p1, p2 = selected[i], selected[i+1]
            c1, c2 = crossover(p1, p2)
            new_pop.extend([c1, c2])
        
        # Elitism
        elites = pop[np.argsort(fitness)[-elite_size:]]
        new_pop.extend(elites)
        
        # Mutation
        for i in range(len(new_pop)):
            new_pop[i] = mutate(new_pop[i], mutation_rate, valid_jobs)
        
        pop = np.array(new_pop)[:pop_size]  # Ensure population size stays constant
        
        print(f"Generation {gen+1}/{generations} | Best Score: {best_score:.2f}", end='\r')
    
    print()  # New line after progress updates
    return best_solution, best_score

# ---------- 4. Main Execution ----------
if __name__ == '__main__':
    # Load and preprocess data
    seekers_df, jobs_df = load_data()
    seekers_df, jobs_df, seeker_skills, job_skills, seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes = preprocess_data(seekers_df, jobs_df)
    
    # Calculate feature matrix
    F = calculate_features(seekers_df, jobs_df, seeker_skills, job_skills, 
                         seek_sector_codes, job_sector_codes, seek_cont_codes, job_cont_codes)
    
    # Define weights for each feature
    WEIGHTS = np.array([0.40, 0.15, 0.15, 0.10, 0.10, 0.05, 0.05], dtype=np.float32)
    
    # Create valid jobs matrix
    valid_jobs = (
    (seekers_df['edu_rank'].values[:, None] >= jobs_df['edu_rank'].values[None, :]) &
    (seekers_df['years_experience'].values[:, None] >= jobs_df['min_experience'].values[None, :]) &
    np.array([
        [len(seeker_skills[i] & job_skills[j]) >= max(1, len(job_skills[j]) // 2) for j in range(len(jobs_df))]
        for i in range(len(seekers_df))
    ])
    )

    
    # Run genetic algorithm
    best_solution, best_score = run_genetic_algorithm(F, WEIGHTS, valid_jobs)
    
    # Display results
    print("\nBest Solution Score:", best_score)
    print("\nOptimal Matches:")
    for i, job_idx in enumerate(best_solution):
        if job_idx >= 0:  # Only show actual matches
            seeker = seekers_df.iloc[i]
            job = jobs_df.iloc[job_idx]
            
            # Calculate match details
            common_skills = seeker_skills[i] & job_skills[job_idx]
            missing_skills = job_skills[job_idx] - seeker_skills[i]
            sector_match = seeker['sector'] == job['sector']
            contract_match = seeker['contract_type'] == job['contract_type']
            
            print(f"\nSeeker {i} → Job {job['job_id']}")
            print(f"  Skills: {len(common_skills)}/{len(job_skills[job_idx])} matched")
            if missing_skills:
                print(f"  Missing Skills: {', '.join(missing_skills)}")
            print(f"  Sector: {'Match' if sector_match else f'Mismatch (Seeker: {seeker["sector"]}, Job: {job["sector"]})'}")
            print(f"  Contract: {'Match' if contract_match else f'Mismatch (Seeker: {seeker["contract_type"]}, Job: {job["contract_type"]})'}")
            print(f"  Experience: {seeker['years_experience']}y (Req: {job['min_experience']}y)")
            print(f"  Education: {seeker['highest_education']} (Req: {job['min_education']})")
            print(f"  Salary: Seeker dza{seeker['salary']:,} vs Job Offer dza{job['salary_offer']:,}")
    
    # Show top candidates per job
    print("\nTop Candidates per Job:")
    for j in range(len(jobs_df)):
        job = jobs_df.iloc[j]
        scores = F[:,j].dot(WEIGHTS)
        valid = valid_jobs[:,j]
        ranked = sorted([(i, scores[i]) for i in np.where(valid)[0]], key=lambda x: -x[1])[:5]
        
        print(f"\nJob {job['job_id']} ({job['sector']}):")
        for rank, (i, score) in enumerate(ranked, 1):
            seeker = seekers_df.iloc[i]
            common_skills = seeker_skills[i] & job_skills[j]
            print(f"{rank}. [Score: {score:.2%}] {seeker['technical_skills']}")
            print(f"   Sector: {seeker['sector']} | Exp: {seeker['years_experience']}y | Edu: {seeker['highest_education']}")
            print(f"   Matching Skills: {', '.join(common_skills)}")

Generation 100/100 | Best Score: 152.41

Best Solution Score: 152.41051496565342

Optimal Matches:

Seeker 1 → Job DATA-002
  Skills: 1/3 matched
  Missing Skills: teamwork, python
  Sector: Mismatch (Seeker: E-Commerce, Job: Data Science)
  Contract: Mismatch (Seeker: Stage, Job: CDD)
  Experience: 6y (Req: 2y)
  Education: Master (Req: Bachelor)
  Salary: Seeker $96,200.0 vs Job Offer $65,000

Seeker 13 → Job DEV-001
  Skills: 1/3 matched
  Missing Skills: communication, hysys
  Sector: Mismatch (Seeker: Aerospace & Defense, Job: Energy & Petroleum)
  Contract: Mismatch (Seeker: Freelance, Job: CDD)
  Experience: 28y (Req: 3y)
  Education: Master (Req: License)
  Salary: Seeker $115,200.0 vs Job Offer $75,000

Seeker 18 → Job DEV-001
  Skills: 2/3 matched
  Missing Skills: hysys
  Sector: Mismatch (Seeker: Retail, Job: Energy & Petroleum)
  Contract: Mismatch (Seeker: Alternance, Job: CDD)
  Experience: 31y (Req: 3y)
  Education: Master (Req: License)
  Salary: Seeker $122,500.0 vs J